# Prolog Program Explainer using LangChain

This notebook creates a tool that provides natural language explanations of Prolog programs using LangChain.

In [ ]:
# Install required packages
!pip install langchain langchain_openai python-dotenv

In [ ]:
# Import necessary libraries
import os
from dotenv import load_dotenv
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Load environment variables (especially API keys)
load_dotenv()

In [ ]:
# Define the output structure
class PrologExplanation(BaseModel):
    summary: str = Field(description="A brief summary of what the Prolog program does")
    predicates: List[dict] = Field(description="List of predicates and their explanations")
    logic_explanation: str = Field(description="How the program logic works")
    examples: List[str] = Field(description="Examples of how the program might be queried and the results")

In [ ]:
# Create an instance of the output parser
parser = PydanticOutputParser(pydantic_object=PrologExplanation)

# Set up the LLM
model = ChatOpenAI(temperature=0, model_name="gpt-4")

# Define the explanation prompt
explanation_template = """
You are a helpful assistant that explains Prolog programs in natural language. 
Given the following Prolog code, explain how it works, what it does, and how it can be used.

Prolog code:
```prolog
{prolog_code}
```

Provide your explanation in a structured format following these guidelines:
{format_instructions}
"""

prompt = PromptTemplate(
    template=explanation_template,
    input_variables=["prolog_code"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# Create the chain
explanation_chain = LLMChain(llm=model, prompt=prompt, output_parser=parser)

In [ ]:
def explain_prolog(prolog_code):
    """Generate a natural language explanation of a Prolog program.
    
    Args:
        prolog_code (str): The Prolog code to explain
        
    Returns:
        PrologExplanation: Structured explanation of the Prolog program
    """
    result = explanation_chain.invoke({"prolog_code": prolog_code})
    return result

## Example Usage

In [ ]:
# Example Prolog code - Family relationships
family_prolog = """
% Facts
parent(john, bob).
parent(john, mary).
parent(jane, bob).
parent(jane, mary).
parent(bob, ann).
parent(bob, pat).
parent(mary, jim).

% Rules
father(X, Y) :- parent(X, Y), male(X).
mother(X, Y) :- parent(X, Y), female(X).
grandparent(X, Z) :- parent(X, Y), parent(Y, Z).
sibling(X, Y) :- parent(P, X), parent(P, Y), X \= Y.

% Gender facts
male(john).
male(bob).
male(jim).
female(jane).
female(mary).
female(ann).
female(pat).
"""

explanation = explain_prolog(family_prolog)
print(f"Summary: {explanation.summary}\n")
print("Predicates:")
for pred in explanation.predicates:
    print(f"- {pred.get('name')}: {pred.get('description')}")
print(f"\nLogic: {explanation.logic_explanation}\n")
print("Examples:")
for example in explanation.examples:
    print(f"- {example}")

In [ ]:
# Example Prolog code - List processing
list_prolog = """
% Append two lists
append([], L, L).
append([H|T], L, [H|R]) :- append(T, L, R).

% Check if element is a member of a list
member(X, [X|_]).
member(X, [_|T]) :- member(X, T).

% Get the length of a list
list_length([], 0).
list_length([_|T], N) :- list_length(T, N1), N is N1 + 1.

% Reverse a list
reverse(L, R) :- reverse_acc(L, [], R).
reverse_acc([], Acc, Acc).
reverse_acc([H|T], Acc, R) :- reverse_acc(T, [H|Acc], R).
"""

explanation = explain_prolog(list_prolog)
print(f"Summary: {explanation.summary}\n")
print("Predicates:")
for pred in explanation.predicates:
    print(f"- {pred.get('name')}: {pred.get('description')}")
print(f"\nLogic: {explanation.logic_explanation}\n")
print("Examples:")
for example in explanation.examples:
    print(f"- {example}")

## Custom Prolog Code Explanation

In [ ]:
# Enter your own Prolog code to explain
custom_prolog = """
% Enter your Prolog code here
"""

# Uncomment and run when you have code to explain
# explanation = explain_prolog(custom_prolog)
# print(f"Summary: {explanation.summary}\n")
# print("Predicates:")
# for pred in explanation.predicates:
#     print(f"- {pred.get('name')}: {pred.get('description')}")
# print(f"\nLogic: {explanation.logic_explanation}\n")
# print("Examples:")
# for example in explanation.examples:
#     print(f"- {example}")